In [1]:
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.patheffects as pe
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

In [2]:
# # load variable datasets
# # shear
# u200 = xr.open_mfdataset(
#     "datasets/ERA5_1940/u200*.nc",
#     combine = "by_coords",
#     preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore"),
# )
# u850 = xr.open_mfdataset(
#     "datasets/ERA5_1940/u850*.nc",
#     combine = "by_coords",
#     preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore"),
# )
# v200 = xr.open_mfdataset(
#     "datasets/ERA5_1940/v200*.nc",
#     combine = "by_coords",
#     preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore"),
# )
# v850 = xr.open_mfdataset(
#     "datasets/ERA5_1940/v850*.nc",
#     combine = "by_coords",
#     preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore"),
# )
# rhum
rhum = xr.open_mfdataset(
    "datasets/ERA5_1940/rhum*.nc",
    combine="by_coords",
    preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore"),
    parallel=True,
    data_vars="minimal",
    coords="minimal",
    compat="override",
)
rhum = rhum[["r"]]

# print(rhum)

# # rename columns
# u200 = u200.rename({
#     "valid_time": "time",
#     "latitude": "lat",
#     "longitude": "lon"
# })
# u850 = u850.rename({
#     "valid_time": "time",
#     "latitude": "lat",
#     "longitude": "lon"
# })
# v200 = v200.rename({
#     "valid_time": "time",
#     "latitude": "lat",
#     "longitude": "lon"
# })
# v850 = v850.rename({
#     "valid_time": "time",
#     "latitude": "lat",
#     "longitude": "lon"
# })
rhum = rhum.rename({
    "valid_time": "time",
    "latitude": "lat",
    "longitude": "lon"
})

print(rhum)

<xarray.Dataset> Size: 11GB
Dimensions:         (time: 13158, pressure_level: 1, lat: 361, lon: 581)
Coordinates:
  * time            (time) datetime64[ns] 105kB 1940-06-01 ... 2025-10-31
  * pressure_level  (pressure_level) float64 8B 600.0
  * lat             (lat) float64 3kB 90.0 89.75 89.5 89.25 ... 0.5 0.25 0.0
  * lon             (lon) float64 5kB -120.0 -119.8 -119.5 ... 24.5 24.75 25.0
    number          int64 8B ...
    expver          (time) object 105kB dask.array<chunksize=(4131,), meta=np.ndarray>
Data variables:
    r               (time, pressure_level, lat, lon) float32 11GB dask.array<chunksize=(1032, 1, 73, 117), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-10-21T23:30 GRIB to CDM+CF via cfgr

/var/folders/d8/dpf8hpcx6l34ky46wf5b_v2h0000gn/T/ipykernel_802/1656455243.py:24: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  rhum = xr.open_mfdataset(


In [3]:
# load land mask
land_ds = xr.open_dataset(
    "datasets/land_mask/land_mask_ERA5.nc"
)

# rename columns
land_ds = land_ds.rename({
    "valid_time": "time",
    "latitude": "lat",
    "longitude": "lon"
})

# sort both spatial coordinates
land_ds = land_ds.sortby("lon").sortby("lat")

# print(land_ds)

# extract land mask and remove dimensions
land = land_ds["lsm"].squeeze(drop=True)

# keep ocean cells where land fraction <= 0.5
ocean_mask = land <= 0.5

# # apply mask to the variable dataset
# u200_ocean = u200.where(ocean_mask)
# u850_ocean = u850.where(ocean_mask)
# v200_ocean = v200.where(ocean_mask)
# v850_ocean = v850.where(ocean_mask)
rhum_ocean = rhum.where(ocean_mask)

In [4]:
# # save file
# u200_ocean.to_netcdf("datasets/u-wind/post_processing/post_landmask/uwnd200_1940-2025_landmasked.nc")
# u850_ocean.to_netcdf("datasets/u-wind/post_processing/post_landmask/uwnd850_1940-2025_landmasked.nc")
# v200_ocean.to_netcdf("datasets/v-wind/post_processing/post_landmask/vwnd200_1940-2025_landmasked.nc")
# v850_ocean.to_netcdf("datasets/v-wind/post_processing/post_landmask/vwnd850_1940-2025_landmasked.nc")
rhum_ocean.to_netcdf("datasets/RHUM/post_processing/post_landmask/rhum600_1940-2025_landmasked_v4.nc")


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 326 in H5FD__sec2_open(): unable to open file: name = '/Users/allisonclanc

KeyboardInterrupt: 

In [5]:
from pathlib import Path

out_dir = Path(
    "datasets/RHUM/post_processing/post_landmask/yearly"
)
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "rhum600_1940_landmasked.nc"

# Select only relative humidity for 1940
yearly = rhum_ocean["r"].sel(
    time=slice("1940-01-01", "1940-12-31")
)

# Smaller chunks to reduce memory use
yearly = yearly.chunk({
    "time": 30,
    "lat": 90,
    "lon": 140
})

print("Dimensions:", dict(yearly.sizes))
print("Valid values:", yearly.count().compute().item())

yearly.to_netcdf(out_file, engine="netcdf4")

print("Saved:", out_file)

Dimensions: {'time': 153, 'pressure_level': 1, 'lat': 361, 'lon': 581}
Valid values: 19535346


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 326 in H5FD__sec2_open(): unable to open file: name = '/Users/allisonclanc

Saved: datasets/RHUM/post_processing/post_landmask/yearly/rhum600_1940_landmasked.nc


In [7]:
from pathlib import Path

out_dir = Path(
    "datasets/RHUM/post_processing/post_landmask/yearly"
)
out_dir.mkdir(parents=True, exist_ok=True)

for year in range(1941, 2026):
    out_file = out_dir / f"rhum600_{year}_landmasked.nc"

    # Select one year and the RH variable
    yearly = rhum_ocean["r"].sel(
        time=slice(f"{year}-01-01", f"{year}-12-31")
    )

    if yearly.sizes.get("time", 0) == 0:
        print(f"Skipping {year}: no data")
        continue

    # Use manageable chunks
    yearly = yearly.chunk({
        "time": 30,
        "lat": 90,
        "lon": 140
    })

    print(f"Saving {year}...")
    yearly.to_netcdf(out_file, engine="netcdf4")
    print(f"Saved {year}")

Saving 1941...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 326 in H5FD__sec2_open(): unable to open file: name = '/Users/allisonclanc

Saved 1941
Saving 1942...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 326 in H5FD__sec2_open(): unable to open file: name = '/Users/allisonclanc

Saved 1942
Saving 1943...


...

Saved 2023
Saving 2024...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 326 in H5FD__sec2_open(): unable to open file: name = '/Users/allisonclanc

Saved 2024
Saving 2025...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 326 in H5FD__sec2_open(): unable to open file: name = '/Users/allisonclanc

Saved 2025
